# NB-16: WanS2VModel -- Complete S2V Forward Pass## Learning Objectives1. **Understand that WanS2VModel is a standalone `nn.Module`**, NOT a WanModel subclass -- it reimplements `patchify()`/`unpatchify()` and has a completely different `forward()` signature with audio, motion, and pose inputs.2. **Trace unified sequence composition:** content (256) + ref (64) + motion (96) = 416 tokens, with `nn.Embedding(3, dim)` mask embeddings distinguishing the three categories.3. **See how mask embeddings** (`nn.Embedding(3, dim)`) assign category 0=content, 1=reference, 2=motion to each token position in the unified sequence.4. **Follow the complete data flow** from latent inputs through embedding, sequence composition, the block loop with dual-timestep and audio injection, to reconstructed output.## Prerequisites- **NB-13** (Audio encoding) -- CausalAudioEncoder output shapes: global `[B, F, 1, dim]`, local `[B, F, num_audio_token+1, dim]`- **NB-14** (FramePackMotioner + per-token RoPE) -- Multi-scale motion token encoding (1x/2x/4x Conv3d patchify)- **NB-15** (WanS2VDiTBlock + AudioInjector) -- Dual-timestep per-position splitting and per-frame audio cross-attention- **NB-07** (WanModel forward pass pattern) -- Base DiT forward pass structure that S2V parallels## Data Flow Diagram```WanS2VModel.forward -- Complete Data Flow============================================INPUTS:  latents      [1, 16, 5, 16, 16]   (ref frame + 4 content frames)  timestep     [1]                   (diffusion timestep, e.g. 0.5)  context      [1, 10, 384]          (text embeddings)  audio_input  [1, 25, 384, 16]      (WAV2Vec features: 25 layers)  motion_lats  [16, 5, 16, 16]       (5 available motion frames)  pose_cond    [1, 16, 4, 16, 16]    (pose skeleton conditioning)STAGE 1 -- SPLIT + EMBEDDINGS:  origin_ref = latents[:,:,0:1]       [1, 16, 1, 16, 16]  (frame 0)  x = latents[:,:,1:]                 [1, 16, 4, 16, 16]  (frames 1-4)  text_embedding(context)             [1, 10, 384]  cal_audio_emb(audio_input):    audio_emb_global                  [1, 4, 1, 384]    merged_audio_emb                  [1, 4, 3, 384]STAGE 2 -- PATCHIFY CONTENT + REFERENCE:  patch_embedding(x) + cond_encoder(pose)    -> patchify -> x_seq              [1, 256, 384]  (content tokens)  patch_embedding(origin_ref)    -> patchify -> ref_seq            [1, 64, 384]   (reference tokens)  cat(x_seq, ref_seq)                 [1, 320, 384]STAGE 3 -- ROPE + MOTION INJECTION:  rope_precompute for content+ref     [1, 320, 4, 48]  (complex)  inject_motion (FramePackMotioner):    + 96 motion tokens                [1, 416, 384]  mask: [0]*256 + [1]*64 + [2]*96     [1, 416]STAGE 4 -- MASK EMBEDDING:  trainable_cond_mask(mask)            [1, 416, 384]  x = x + mask_emb                    [1, 416, 384]STAGE 5 -- DUAL TIMESTEP:  cat([0.5, 0.0]) -> sinusoidal -> time_embedding -> t          [2, 384]  time_projection -> unflatten -> unsqueeze+transpose -> t_mod  [1, 6, 2, 384]STAGE 6 -- TRANSFORMER BLOCKS:  3x WanS2VDiTBlock (dual-timestep per-position split)  + after_transformer_block (audio injection at layers [0, 2])  Shape preserved: [1, 416, 384]STAGE 7 -- OUTPUT:  Slice content tokens:  [1, 416, 384] -> [1, 256, 384]  Head(x, t[:-1]):       [1, 256, 384] -> [1, 256, 64]  unpatchify:            [1, 256, 64]  -> [1, 16, 4, 16, 16]  cat(ref, video):       -> [1, 16, 5, 16, 16]OUTPUT: [1, 16, 5, 16, 16]  (matches input shape)```

In [ ]:
# Source: wan_video_dit_s2v.py -- S2V module loading setupimport sys, types, importlib.util, pathlibimport torchimport torch.nn as nnimport torch.nn.functional as F# Find project root: walk up from Course/ to find the directory containing diffsynth/# This handles both normal checkout and git worktree scenarios._here = pathlib.Path().resolve()PROJECT_ROOT = Nonefor _candidate in [_here] + list(_here.parents)[:6]:    if (_candidate / "diffsynth" / "models" / "wan_video_dit.py").exists():        PROJECT_ROOT = _candidate        breakif PROJECT_ROOT is None:    raise FileNotFoundError(        "Could not find diffsynth/models/wan_video_dit.py. "        "Run this notebook from Course/ inside the project checkout."    )print(f"Project root: {PROJECT_ROOT}")# Stub wan_video_camera_controller (not needed for S2V forward pass demos)_cam_stub = types.ModuleType("diffsynth.models.wan_video_camera_controller")_cam_stub.SimpleAdapter = type("SimpleAdapter", (), {"__init__": lambda s, *a, **kw: None})sys.modules["diffsynth"] = types.ModuleType("diffsynth")sys.modules["diffsynth.models"] = types.ModuleType("diffsynth.models")sys.modules["diffsynth.models.wan_video_camera_controller"] = _cam_stub# Load wan_video_dit.py (base DiT module, dependency of S2V)_dit_path = PROJECT_ROOT / "diffsynth" / "models" / "wan_video_dit.py"_spec = importlib.util.spec_from_file_location("diffsynth.models.wan_video_dit", _dit_path)dit_mod = importlib.util.module_from_spec(_spec)sys.modules["diffsynth.models.wan_video_dit"] = dit_mod_spec.loader.exec_module(dit_mod)# Load wan_video_dit_s2v.py (S2V model with all S2V classes)_s2v_path = PROJECT_ROOT / "diffsynth" / "models" / "wan_video_dit_s2v.py"_spec = importlib.util.spec_from_file_location("diffsynth.models.wan_video_dit_s2v", _s2v_path)s2v_mod = importlib.util.module_from_spec(_spec)sys.modules["diffsynth.models.wan_video_dit_s2v"] = s2v_mod_spec.loader.exec_module(s2v_mod)from diffsynth.models.wan_video_dit_s2v import WanS2VModelfrom diffsynth.models.wan_video_dit import sinusoidal_embedding_1dprint(f"Imported: WanS2VModel, sinusoidal_embedding_1d")print("Setup complete.")

In [ ]:
# Source: wan_video_dit_s2v.py, line 359 (WanS2VModel.__init__ parameters)# Reduced config (per D-09)dim = 384num_heads = 4head_dim = dim // num_heads  # 96ffn_dim = 1024freq_dim = 256patch_size = (1, 2, 2)in_dim = 16       # VAE latent channelsout_dim = in_dim  # Head internally multiplies by prod(patch_size): 16 * 4 = 64 per tokentext_dim = 384    # text embedding input dim (simplified)eps = 1e-6# S2V-specific config (per D-06 through D-08)zip_frame_buckets = [1, 2, 16]     # WanS2VModel hardcodes these (production values)audio_inject_layers = [0, 2]num_layers = 3num_audio_token = 2audio_dim = 384cond_dim = 16     # pose conditioning channels (same as in_dim)# Input dimensionsF_content = 4     # content frameslat_height, lat_width = 16, 16video_length = 16  # audio temporal lengthnum_wav2vec_layers = 25  # CausalAudioEncoder default (production value)# Derived sequence lengthsH_patch = lat_height // patch_size[1]   # 8W_patch = lat_width // patch_size[2]    # 8S_content = F_content * H_patch * W_patch  # 4 * 8 * 8 = 256S_ref = 1 * H_patch * W_patch  # 1 * 8 * 8 = 64# Motion tokens: WanS2VModel uses zip_frame_buckets=[1, 2, 16]# 1x: Conv3d(k=(1,2,2), s=(1,2,2)) -> 1*8*8 = 64# 2x: Conv3d(k=(2,4,4), s=(2,4,4)) -> 1*4*4 = 16# 4x: Conv3d(k=(4,8,8), s=(4,8,8)) -> 4*2*2 = 16S_motion = 64 + 16 + 16  # = 96S_total = S_content + S_ref + S_motion  # 256 + 64 + 96 = 416print(f"Sequence: content={S_content} + ref={S_ref} + motion={S_motion} = total={S_total}")print(f"\nNote: WanS2VModel hardcodes zip_frame_buckets=[1, 2, 16]")print(f"  NB-14 used reduced [1, 2, 4] for standalone demo (84 motion tokens)")print(f"  WanS2VModel constructor passes [1, 2, 16] to FramePackMotioner (96 motion tokens)")

## 1. WanS2VModel is NOT a WanModel SubclassUnlike VaceWanAttentionBlock (which inherits DiTBlock, NB-11), WanS2VModel inherits from`torch.nn.Module` directly:```pythonclass WanS2VModel(torch.nn.Module):   # NOT WanModel!```It shares COMPONENTS with WanModel (same DiTBlock/Head/patch_embedding pattern) but:- **Reimplements `patchify()` and `unpatchify()`** with slightly different signatures (no control_adapter handling)- **Has a completely different `forward()` signature** adding `audio_input`, `motion_latents`, `pose_cond`- **Adds 5 S2V-specific modules:** `cond_encoder`, `casual_audio_encoder`, `audio_injector`, `frame_packer`, `trainable_cond_mask`This is an important architectural distinction from VACE (Phase 4), where VaceWanModel IS a WanModel subclass.```WanModel (NB-07)          VaceWanModel (NB-12)      WanS2VModel (this notebook)  torch.nn.Module           WanModel                  torch.nn.Module  (base DiT)                (inherits)                (standalone, NOT subclass)```Source: `diffsynth/models/wan_video_dit_s2v.py`, line 359

In [ ]:
# Source: wan_video_dit_s2v.py, line 359# Sub-Module Inventory Table## | Sub-module             | Type              | Shape/Config               | Role                            | Shared with WanModel? |# |------------------------|-------------------|----------------------------|---------------------------------|-----------------------|# | patch_embedding        | Conv3d            | (16, 384, k=(1,2,2))       | Patchify latent input           | Yes (NB-07)           |# | text_embedding         | Sequential        | (384 -> 384 -> 384)        | Text conditioning MLP           | Yes (NB-07)           |# | time_embedding         | Sequential        | (256 -> 384 -> 384)        | Timestep MLP                    | Yes (NB-07)           |# | time_projection        | Sequential        | (384 -> 2304)              | 6-param modulation projection   | Yes (NB-07)           |# | blocks                 | ModuleList        | 3 x WanS2VDiTBlock         | Transformer blocks              | Modified (S2V var.)   |# | head                   | Head              | (384, 16, (1,2,2))         | Output projection + unpatchify  | Yes (NB-07)           |# | freqs                  | Tensor            | precompute_freqs_cis_3d    | RoPE frequencies                | Yes (NB-07)           |# | cond_encoder           | Conv3d            | (16, 384, k=(1,2,2))       | Pose conditioning patchify      | **NEW (S2V)**         |# | casual_audio_encoder   | CausalAudioEncoder| (384, 25 layers, 384, 2tk) | Audio feature compression(NB13) | **NEW (S2V)**         |# | audio_injector         | AudioInjector_WAN | (384, 4 heads, at [0,2])   | Per-frame audio cross-attn(NB15)| **NEW (S2V)**         |# | frame_packer           | FramePackMotioner | (384, 4 heads, [1,2,16])   | Multi-scale motion patch (NB14) | **NEW (S2V)**         |# | trainable_cond_mask    | Embedding         | (3, 384)                   | Mask: 0=content, 1=ref, 2=motion| **NEW (S2V)**         |# Source: wan_video_dit_s2v.py, line 359model = WanS2VModel(    dim=dim,    in_dim=in_dim,    ffn_dim=ffn_dim,    out_dim=out_dim,    text_dim=text_dim,    freq_dim=freq_dim,    eps=eps,    patch_size=patch_size,    num_heads=num_heads,    num_layers=num_layers,    cond_dim=cond_dim,    audio_dim=audio_dim,    num_audio_token=num_audio_token,    enable_adain=True,    audio_inject_layers=audio_inject_layers,    zero_timestep=True,    add_last_motion=True,    framepack_drop_mode="padd",)model.eval()total_params = sum(p.numel() for p in model.parameters())print(f"WanS2VModel: {total_params:,} parameters")# Verify it is NOT a WanModel subclassfrom diffsynth.models.wan_video_dit import WanModelassert not isinstance(model, WanModel), "WanS2VModel should NOT be a WanModel subclass"assert isinstance(model, torch.nn.Module), "WanS2VModel must be a torch.nn.Module"print(f"isinstance(model, WanModel): {isinstance(model, WanModel)}  (NOT a subclass!)")print(f"isinstance(model, nn.Module): {isinstance(model, torch.nn.Module)}")# Verify S2V-specific modules existassert hasattr(model, 'cond_encoder'), "Missing cond_encoder"assert hasattr(model, 'casual_audio_encoder'), "Missing casual_audio_encoder"assert hasattr(model, 'audio_injector'), "Missing audio_injector"assert hasattr(model, 'frame_packer'), "Missing frame_packer"assert hasattr(model, 'trainable_cond_mask'), "Missing trainable_cond_mask"print(f"\nAll 5 S2V-specific modules verified present")# Verify trainable_cond_mask has 3 categoriesassert model.trainable_cond_mask.num_embeddings == 3assert model.trainable_cond_mask.embedding_dim == dimprint(f"trainable_cond_mask: Embedding({model.trainable_cond_mask.num_embeddings}, {model.trainable_cond_mask.embedding_dim})")print(f"  0 = content, 1 = reference, 2 = motion")# Verify Head internally multiplies out_dim by prod(patch_size)expected_head_out = out_dim * patch_size[0] * patch_size[1] * patch_size[2]assert model.head.head.out_features == expected_head_outprint(f"\nHead linear: {model.head.head.in_features} -> {model.head.head.out_features}")print(f"  = dim -> out_dim * prod(patch_size) = {dim} -> {out_dim} * {patch_size[0]*patch_size[1]*patch_size[2]} = {expected_head_out}")

In [ ]:
# Source: wan_video_dit_s2v.py, line 503 (forward signature)B = 1# latents: ref frame (1) + content frames (F_content) = F_content + 1latents = torch.randn(B, in_dim, F_content + 1, lat_height, lat_width)timestep = torch.tensor([0.5])context = torch.randn(B, 10, text_dim)  # 10 text tokensaudio_input = torch.randn(B, num_wav2vec_layers, audio_dim, video_length)motion_latents = [torch.randn(in_dim, 5, lat_height, lat_width)]  # 5 available motion framespose_cond = torch.randn(B, cond_dim, F_content, lat_height, lat_width)print(f"=== Input Tensors ===")print(f"latents:         {latents.shape}  (1 ref + {F_content} content frames)")print(f"timestep:        {timestep.shape}  value={timestep.item()}")print(f"context:         {context.shape}  (text embeddings)")print(f"audio_input:     {audio_input.shape}  ({num_wav2vec_layers} WAV2Vec layers)")print(f"motion_latents:  {motion_latents[0].shape}  (5 available frames)")print(f"pose_cond:       {pose_cond.shape}  (pose skeleton)")assert latents.shape == torch.Size([B, in_dim, F_content + 1, lat_height, lat_width])assert audio_input.shape[1] == num_wav2vec_layers, f"audio_input needs {num_wav2vec_layers} layers"assert pose_cond.shape == torch.Size([B, cond_dim, F_content, lat_height, lat_width])

## 2. Step-by-Step Forward PassWe now trace `WanS2VModel.forward()` stage by stage, following the data flow diagram above.Each stage shows the exact shapes and operations from the source code.**Note on motion injection:** The default `forward()` passes `drop_motion_frames=True` to`inject_motion()`, which means motion tokens are NOT included during standard inference.We trace with `drop_motion_frames=False` here to demonstrate the full architecture includingmotion tokens, since this reveals the complete 3-category mask mechanism (0=content, 1=ref, 2=motion).

In [ ]:
# Source: wan_video_dit_s2v.py, lines 514-521# ---- STAGE 1: SPLIT + EMBEDDINGS ----with torch.no_grad():    # Split reference frame from content frames    origin_ref = latents[:, :, 0:1]    x = latents[:, :, 1:]    print(f"Stage 1a -- Split latents:")    print(f"  origin_ref: {origin_ref.shape}  (frame 0 = reference)")    print(f"  x (content): {x.shape}  (frames 1-{F_content})")    assert origin_ref.shape == torch.Size([B, in_dim, 1, lat_height, lat_width])    assert x.shape == torch.Size([B, in_dim, F_content, lat_height, lat_width])    # Text embedding    # Source: wan_video_dit_s2v.py, line 518    ctx = model.text_embedding(context)    print(f"\nStage 1b -- text_embedding: {context.shape} -> {ctx.shape}")    assert ctx.shape == torch.Size([B, 10, dim])    # Audio encoding (cal_audio_emb)    # Source: wan_video_dit_s2v.py, line 521    # cal_audio_emb prepends motion_frames[0]=73 copies of first timestep,    # runs through CausalAudioEncoder (with 4x temporal downsample),    # then slices from motion_frames[1]=19 onwards.    audio_emb_global, merged_audio_emb = model.cal_audio_emb(audio_input)    print(f"\nStage 1c -- cal_audio_emb:")    print(f"  audio_emb_global:  {audio_emb_global.shape}  (global path, for AdaLayerNorm)")    print(f"  merged_audio_emb:  {merged_audio_emb.shape}  (local path, for CrossAttention)")    assert audio_emb_global.shape[2] == 1, "Global path: single token per frame"    assert merged_audio_emb.shape[2] == num_audio_token + 1, f"Local path: {num_audio_token + 1} tokens per frame"    print(f"\n  audio_emb has {audio_emb_global.shape[1]} temporal frames = F_content = {F_content}")

In [ ]:
# Source: wan_video_dit_s2v.py, lines 523-531# ---- STAGE 2: PATCHIFY CONTENT + REFERENCE ----with torch.no_grad():    # Content: patch_embedding(x) + cond_encoder(pose_cond)    # Source: wan_video_dit_s2v.py, line 525    pose_cond_safe = torch.zeros_like(x) if pose_cond is None else pose_cond    x_patched = model.patch_embedding(x) + model.cond_encoder(pose_cond_safe)    print(f"Stage 2a -- patch_embedding(x) + cond_encoder(pose):")    print(f"  x: {x.shape} -> patch_embedding -> {model.patch_embedding(x).shape}")    print(f"  + cond_encoder(pose): {pose_cond.shape} -> {model.cond_encoder(pose_cond_safe).shape}")    print(f"  combined: {x_patched.shape}")    # Patchify: flatten spatial dims    # Source: wan_video_dit_s2v.py, line 424    x_seq, (f, h, w) = model.patchify(x_patched)    seq_len_x = x_seq.shape[1]    print(f"\nStage 2b -- patchify (flatten):")    print(f"  {x_patched.shape} -> {x_seq.shape}  grid=({f},{h},{w})")    assert x_seq.shape == torch.Size([B, S_content, dim])    assert seq_len_x == S_content  # 256    # Reference image: same patch_embedding, separate patchify    # Source: wan_video_dit_s2v.py, line 529    ref_patched = model.patch_embedding(origin_ref)    ref_seq, (rf, rh, rw) = model.patchify(ref_patched)    print(f"\nStage 2c -- reference patchify:")    print(f"  {origin_ref.shape} -> patch_embedding -> {ref_patched.shape} -> patchify -> {ref_seq.shape}")    assert ref_seq.shape == torch.Size([B, S_ref, dim])    # Concatenate content + reference    # Source: wan_video_dit_s2v.py, line 531    x_combined = torch.cat([x_seq, ref_seq], dim=1)    print(f"\nStage 2d -- cat(content, ref):")    print(f"  {x_seq.shape} + {ref_seq.shape} -> {x_combined.shape}")    assert x_combined.shape == torch.Size([B, S_content + S_ref, dim])  # [1, 320, 384]

In [ ]:
# Source: wan_video_dit_s2v.py, lines 530-539# ---- STAGE 3: ROPE + MOTION INJECTION ----from diffsynth.models.wan_video_dit_s2v import rope_precomputewith torch.no_grad():    # Get grid sizes for content + reference RoPE    # Source: wan_video_dit_s2v.py, line 530    grid_sizes = model.get_grid_sizes((f, h, w), (rf, rh, rw))    # Initial mask: 0 for content, 1 for reference    # Source: wan_video_dit_s2v.py, line 533    mask = torch.cat([        torch.zeros([1, seq_len_x]),        torch.ones([1, ref_seq.shape[1]])    ], dim=1).to(torch.long)    print(f"Stage 3a -- initial mask: {mask.shape}  (0=content, 1=ref)")    # Compute content+ref RoPE    # Source: wan_video_dit_s2v.py, lines 535-537    pre_freqs = rope_precompute(        x_combined.detach().view(1, x_combined.size(1), num_heads, dim // num_heads),        grid_sizes, model.freqs, start=None    )    print(f"Stage 3b -- content+ref RoPE: {pre_freqs.shape}")    # Inject motion tokens (FramePackMotioner)    # Source: wan_video_dit_s2v.py, line 539    # NOTE: We use drop_motion_frames=False to demonstrate motion injection.    # The default forward() uses drop_motion_frames=True (no motion tokens).    x_full, pre_freqs, mask = model.inject_motion(        x_combined, pre_freqs, mask, motion_latents,        drop_motion_frames=False, add_last_motion=2    )    print(f"\nStage 3c -- inject_motion:")    print(f"  {x_combined.shape} + motion -> {x_full.shape}")    print(f"  mask: {mask.shape}  unique values: {mask.unique().tolist()}")    actual_motion = x_full.shape[1] - S_content - S_ref    assert x_full.shape[1] == S_total, f"Expected {S_total} total tokens, got {x_full.shape[1]}"    assert set(mask.unique().tolist()) == {0, 1, 2}, "Mask should have 0(content), 1(ref), 2(motion)"print(f"\nUnified sequence: {S_content} content + {S_ref} ref + {actual_motion} motion = {x_full.shape[1]}")

In [ ]:
# Source: wan_video_dit_s2v.py, line 541# ---- STAGE 4: MASK EMBEDDING ----with torch.no_grad():    mask_emb = model.trainable_cond_mask(mask)    print(f"Stage 4 -- mask embedding:")    print(f"  mask: {mask.shape} (values: 0/1/2)")    print(f"  trainable_cond_mask(mask): {mask_emb.shape}")    assert mask_emb.shape == torch.Size([1, S_total, dim])    x_full = x_full + mask_emb.to(x_full.dtype)    print(f"  x = x + mask_emb: {x_full.shape}")    # Show token category distribution    n_content = (mask == 0).sum().item()    n_ref = (mask == 1).sum().item()    n_motion = (mask == 2).sum().item()    print(f"\n  Content tokens (mask=0): {n_content}")    print(f"  Reference tokens (mask=1): {n_ref}")    print(f"  Motion tokens (mask=2): {n_motion}")    assert n_content == S_content    assert n_ref == S_ref    assert n_motion == S_motion

In [ ]:
# Source: wan_video_dit_s2v.py, lines 544-546# ---- STAGE 5: DUAL TIMESTEP ----with torch.no_grad():    # Source: wan_video_dit_s2v.py, line 544    timestep_dual = torch.cat([timestep, torch.zeros([1], dtype=timestep.dtype)])    t = model.time_embedding(sinusoidal_embedding_1d(model.freq_dim, timestep_dual))    # Source: wan_video_dit_s2v.py, line 546    t_mod = model.time_projection(t).unflatten(1, (6, model.dim)).unsqueeze(2).transpose(0, 2)    print(f"Stage 5 -- dual timestep:")    print(f"  timestep: {timestep.tolist()} -> cat with 0 -> {timestep_dual.tolist()}")    print(f"  time_embedding: {timestep_dual.shape} -> {t.shape}")    print(f"  time_projection + reshape: {t.shape} -> {t_mod.shape}")    assert t_mod.shape == torch.Size([1, 6, 2, dim])    print(f"  t_mod[:,:,0,:] = real timestep, t_mod[:,:,1,:] = zero timestep")    print(f"\n  (See NB-15 for detailed dual-timestep per-position trace)")

In [ ]:
# Source: wan_video_dit_s2v.py, lines 553-587# ---- STAGE 6: TRANSFORMER BLOCKS ----with torch.no_grad():    x_running = x_full.clone()    print(f"Stage 6 -- block loop ({num_layers} blocks):")    print(f"  Input: {x_running.shape}")    print(f"  audio_inject_layers: {audio_inject_layers}\n")    for block_id, block in enumerate(model.blocks):        # WanS2VDiTBlock forward (dual-timestep per-position split)        # Source: wan_video_dit_s2v.py, line 586        x_running = block(x_running, ctx, t_mod, seq_len_x, pre_freqs[0])        # Audio injection after specified blocks        # Source: wan_video_dit_s2v.py, line 587        x_running = model.after_transformer_block(            block_id, x_running, audio_emb_global, merged_audio_emb, seq_len_x        )        injected = "  + AUDIO INJECTION" if block_id in model.audio_injector.injected_block_id else ""        print(f"  Block {block_id}: {x_running.shape}{injected}")        assert x_running.shape == torch.Size([1, S_total, dim])    print(f"\n  Output after all blocks: {x_running.shape}")

In [ ]:
# Source: wan_video_dit_s2v.py, lines 589-594# ---- STAGE 7: OUTPUT ----with torch.no_grad():    # Slice content tokens only (drop ref + motion)    # Source: wan_video_dit_s2v.py, line 589    x_content = x_running[:, :seq_len_x]    print(f"Stage 7a -- slice content: {x_running.shape} -> {x_content.shape}")    print(f"  Dropped: {S_ref} ref + {S_motion} motion tokens")    assert x_content.shape == torch.Size([B, S_content, dim])    # Head: project to patch space (uses REAL timestep only, not zero)    # Source: wan_video_dit_s2v.py, line 590    x_head = model.head(x_content, t[:-1])    head_out_dim = out_dim * patch_size[0] * patch_size[1] * patch_size[2]    print(f"\nStage 7b -- Head(x, t[:-1]): {x_content.shape} -> {x_head.shape}")    print(f"  Uses t[:-1] = real timestep only (not the zero timestep)")    assert x_head.shape == torch.Size([B, S_content, head_out_dim])  # [1, 256, 64]    # Unpatchify: reshape back to video    # Source: wan_video_dit_s2v.py, line 591    x_video = model.unpatchify(x_head, (f, h, w))    print(f"\nStage 7c -- unpatchify: {x_head.shape} -> {x_video.shape}")    assert x_video.shape == torch.Size([B, in_dim, F_content, lat_height, lat_width])    # Concatenate reference frame back    # Source: wan_video_dit_s2v.py, line 593    output = torch.cat([origin_ref, x_video], dim=2)    print(f"\nStage 7d -- cat(ref, video): {origin_ref.shape} + {x_video.shape} -> {output.shape}")    assert output.shape == torch.Size([B, in_dim, F_content + 1, lat_height, lat_width])    assert output.shape == latents.shape, f"Output shape {output.shape} must match input {latents.shape}"print(f"\nFinal output: {output.shape}  (matches input latents shape!)")

In [ ]:
# Source: wan_video_dit_s2v.py, line 503# Verify direct model call produces the correct output shape.# Note: model.forward() uses drop_motion_frames=True by default,# so it runs without motion tokens (mask has only 0 and 1).# The output shape is the same either way.with torch.no_grad():    output_direct = model(        latents=latents,        timestep=timestep,        context=context,        audio_input=audio_input,        motion_latents=motion_latents,        pose_cond=pose_cond,    )assert output_direct.shape == torch.Size([B, in_dim, F_content + 1, lat_height, lat_width])print(f"Direct model.forward(): {output_direct.shape}")print(f"Output matches input shape: {output_direct.shape == latents.shape}")print(f"\nWanS2VModel forward pass VERIFIED")

## 3. Shape Trace Summary| Step | Operation | Input Shape | Output Shape ||------|-----------|-------------|------------- || 1a | Split ref/content | `[1,16,5,16,16]` | ref:`[1,16,1,16,16]`, x:`[1,16,4,16,16]` || 1b | text_embedding | `[1,10,384]` | `[1,10,384]` || 1c | cal_audio_emb | `[1,25,384,16]` | global:`[1,4,1,384]`, local:`[1,4,3,384]` || 2a | patch_embedding + cond_encoder | `[1,16,4,16,16]` | `[1,384,4,8,8]` || 2b | patchify (flatten) | `[1,384,4,8,8]` | `[1,256,384]` || 2c | ref patchify | `[1,16,1,16,16]` | `[1,64,384]` || 2d | cat(content, ref) | `[1,256,384]`+`[1,64,384]` | `[1,320,384]` || 3c | inject_motion | `[1,320,384]` | `[1,416,384]` || 4 | mask embedding | mask `[1,416]` | `[1,416,384]` || 5 | dual timestep | `[2]` | `[1,6,2,384]` || 6 | 3x blocks + audio inject | `[1,416,384]` | `[1,416,384]` || 7a | slice content | `[1,416,384]` | `[1,256,384]` || 7b | Head | `[1,256,384]` | `[1,256,64]` || 7c | unpatchify | `[1,256,64]` | `[1,16,4,16,16]` || 7d | cat ref back | `[1,16,1,16,16]`+`[1,16,4,16,16]` | `[1,16,5,16,16]` |**Sequence length:** With `F_content=4` and `patch_size=(1,2,2)` on 16x16 spatial: content = 4*8*8 = 256, ref = 1*8*8 = 64, motion = 96 (from FramePackMotioner with `zip_frame_buckets=[1,2,16]`). Total = 416 tokens.**Head output:** `out_dim * prod(patch_size) = 16 * 4 = 64` per token. The `out_dim=16` is passed to the constructor; Head internally multiplies by `prod(patch_size)` to produce the per-token output that unpatchify rearranges back to video shape.

## Summary### Key Takeaways1. **WanS2VModel is a standalone `nn.Module`** that shares components with WanModel (patch_embedding, text_embedding, time_embedding, Head) but is NOT a subclass. It reimplements patchify/unpatchify and has a different forward signature with audio, motion, and pose inputs.2. **The unified sequence composes content + reference + motion tokens** (256 + 64 + 96 = 416), with `nn.Embedding(3, dim)` mask embeddings distinguishing the three categories (0=content, 1=ref, 2=motion). The default forward drops motion tokens (`drop_motion_frames=True`), reducing the sequence to 320.3. **The block loop applies WanS2VDiTBlock** (dual-timestep: content gets real timestep, ref+motion get zero timestep) followed by `after_transformer_block` (audio injection via AdaLayerNorm + CrossAttention at layers [0, 2]).4. **Output reconstruction slices content tokens only**, applies Head with the real timestep (`t[:-1]`), unpatchifies back to video shape, and concatenates the reference frame to produce `[B, 16, F+1, H, W]` matching the input.### Source References| Symbol | Location ||--------|----------|| WanS2VModel | `wan_video_dit_s2v.py`, line 359 || WanS2VModel.forward | `wan_video_dit_s2v.py`, line 503 || patchify / unpatchify | `wan_video_dit_s2v.py`, lines 424, 429 || inject_motion | `wan_video_dit_s2v.py`, line 448 || after_transformer_block | `wan_video_dit_s2v.py`, line 459 || cal_audio_emb | `wan_video_dit_s2v.py`, line 484 || get_grid_sizes | `wan_video_dit_s2v.py`, line 491 |### NextNB-17 (Phase 6) shows how the production pipeline dispatches between VACE and S2V model paths, runs the denoising loop, and applies VAE decode.

## Exercises### Exercise 1 -- Adding a fourth mask categoryThe mask has 3 categories (0=content, 1=ref, 2=motion). What would happen if you added a 4th category for a new conditioning signal (e.g., depth maps)? What would need to change in `trainable_cond_mask` and `inject_motion`?**Hint:** `trainable_cond_mask` would become `nn.Embedding(4, dim)` instead of `nn.Embedding(3, dim)`. You would also need a new patchify projection (like `cond_encoder`) and a new injection method (like `inject_motion`) that appends the depth tokens and assigns them mask value 3.### Exercise 2 -- Why does Head use t[:-1]?In Stage 7, the Head uses `t[:-1]` (real timestep only). Why not `t` (both timesteps)? What does the Head's modulation need to know about, and why doesn't zero-timestep information help?**Hint:** The Head produces a noise prediction for content tokens that ARE noisy. It needs the noise level (real timestep) to properly scale its output. The zero timestep was only used for ref+motion tokens in the block loop -- those tokens are already discarded before the Head runs. Head takes a single `t` of shape `[1, dim]`, not dual timestep.### Exercise 3 -- Compare forward signaturesCompare `WanS2VModel.forward()` signature vs `WanModel.forward()` (from NB-07). List all the additional inputs S2V requires. Why is each one necessary for sketch-to-video generation?**Hint:** S2V adds `audio_input` (lip-sync and sound-reactive generation), `motion_latents` (temporal context from previous frames via FramePack), and `pose_cond` (skeleton conditioning for controllable human pose). WanModel's base `forward()` only takes `(x, timestep, context)` -- the three S2V additions enable multi-modal conditioning beyond text.